# 01 — Data Cleaning

**Purpose:** Prepare raw build-system metrics for downstream analysis.  
Steps include missing-data imputation, outlier detection, path alignment,
target validation against the dependency graph, type casting, and
normalisation. Cleaned datasets are saved back to Parquet.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from build_optimiser.graph import load_graph, attach_metrics
from build_optimiser.config import load_config

cfg = load_config(PROJECT_ROOT / "config.yaml")

# Load processed data
file_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "file_metrics.parquet")
target_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "target_metrics.parquet")
edge_list = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "edge_list.parquet")

# Load dependency graph
G = load_graph(str(PROJECT_ROOT / "data" / "raw" / "dot"))
attach_metrics(G, target_metrics)

print(f"Files: {len(file_metrics)}, Targets: {len(target_metrics)}, Edges: {len(edge_list)}")

In [ ]:
# ── Missing data analysis ─────────────────────────────────────────────
print("=== Missing values per column (target_metrics) ===")
tm_missing = target_metrics.isnull().sum()
print(tm_missing[tm_missing > 0].sort_values(ascending=False))
print()

print("=== Missing values per column (file_metrics) ===")
fm_missing = file_metrics.isnull().sum()
print(fm_missing[fm_missing > 0].sort_values(ascending=False))
print()

# Targets with missing compile times
if "compile_time_s" in target_metrics.columns:
    missing_ct = target_metrics[target_metrics["compile_time_s"].isna()]
    print(f"Targets missing compile_time_s: {len(missing_ct)} / {len(target_metrics)}")
    if len(missing_ct) > 0:
        print(missing_ct.head(10))
else:
    print("Column 'compile_time_s' not found — check schema.")

# Visualise missingness pattern
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(target_metrics.isnull().T, cbar=False, cmap="viridis", ax=ax)
ax.set_title("Missing-value pattern (target_metrics)")
ax.set_xlabel("Row index")
plt.tight_layout()
plt.show()

In [ ]:
# ── Outlier detection (IQR method on compile times) ───────────────────
def flag_iqr_outliers(series: pd.Series, k: float = 1.5) -> pd.Series:
    """Return boolean mask where True = outlier."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return (series < lower) | (series > upper)

if "compile_time_s" in target_metrics.columns:
    ct = target_metrics["compile_time_s"].dropna()
    outlier_mask = flag_iqr_outliers(ct)
    n_outliers = outlier_mask.sum()
    print(f"IQR outliers (k=1.5): {n_outliers} / {len(ct)} ({100*n_outliers/len(ct):.1f}%)")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].boxplot(ct, vert=False)
    axes[0].set_xlabel("compile_time_s")
    axes[0].set_title("Box plot (before cleaning)")

    axes[1].hist(ct[~outlier_mask], bins=50, edgecolor="black", alpha=0.7, label="inliers")
    axes[1].hist(ct[outlier_mask], bins=50, edgecolor="black", alpha=0.7, color="red", label="outliers")
    axes[1].set_xlabel("compile_time_s")
    axes[1].set_ylabel("count")
    axes[1].set_title("Histogram with outliers highlighted")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    # Tag outliers — keep rows but add a flag column
    target_metrics["compile_time_outlier"] = False
    target_metrics.loc[ct.index[outlier_mask], "compile_time_outlier"] = True
else:
    print("Skipping outlier detection — compile_time_s not available.")

In [ ]:
# ── Path alignment verification ──────────────────────────────────────
# Ensure all file paths in file_metrics use POSIX separators and are
# relative to the project root stored in the config.

path_col = "path" if "path" in file_metrics.columns else file_metrics.columns[0]
print(f"Using column '{path_col}' as file-path identifier.")

# Normalise to forward slashes
file_metrics[path_col] = file_metrics[path_col].astype(str).str.replace("\\\\", "/", regex=False)

# Strip any leading './' or absolute prefix matching project root
source_root = cfg.get("source_root", "")
if source_root:
    file_metrics[path_col] = file_metrics[path_col].str.replace(
        f"^{source_root}/?", "", regex=True
    )
file_metrics[path_col] = file_metrics[path_col].str.lstrip("./")

print(f"Sample paths after alignment:")
print(file_metrics[path_col].head(10).to_string())

In [ ]:
# ── Target validation (cross-reference DOT files with build tree) ────
graph_targets = set(G.nodes())
name_col = "target" if "target" in target_metrics.columns else target_metrics.columns[0]
metric_targets = set(target_metrics[name_col].astype(str))

in_graph_not_metrics = graph_targets - metric_targets
in_metrics_not_graph = metric_targets - graph_targets

print(f"Targets in graph only      : {len(in_graph_not_metrics)}")
print(f"Targets in metrics only    : {len(in_metrics_not_graph)}")
print(f"Targets in both            : {len(graph_targets & metric_targets)}")

if in_graph_not_metrics:
    print("\nSample graph-only targets:")
    for t in sorted(in_graph_not_metrics)[:10]:
        print(f"  {t}")

if in_metrics_not_graph:
    print("\nSample metrics-only targets (will be dropped):")
    for t in sorted(in_metrics_not_graph)[:10]:
        print(f"  {t}")

# Drop rows from target_metrics that are not present in the graph
before = len(target_metrics)
target_metrics = target_metrics[target_metrics[name_col].astype(str).isin(graph_targets)].copy()
print(f"\nDropped {before - len(target_metrics)} orphan targets from target_metrics.")

In [ ]:
# ── Type casting and normalisation ───────────────────────────────────
# Ensure numeric columns are proper dtypes and normalise key metrics.

numeric_candidates = [
    "compile_time_s", "link_time_s", "sloc", "header_count",
    "header_depth", "object_size_bytes",
]

for col in numeric_candidates:
    if col in target_metrics.columns:
        target_metrics[col] = pd.to_numeric(target_metrics[col], errors="coerce")

for col in numeric_candidates:
    if col in file_metrics.columns:
        file_metrics[col] = pd.to_numeric(file_metrics[col], errors="coerce")

# Min-max normalisation for selected columns (store as new _norm cols)
norm_cols = [c for c in ["compile_time_s", "sloc", "header_depth"] if c in target_metrics.columns]
for col in norm_cols:
    series = target_metrics[col]
    lo, hi = series.min(), series.max()
    if hi - lo > 0:
        target_metrics[f"{col}_norm"] = (series - lo) / (hi - lo)
    else:
        target_metrics[f"{col}_norm"] = 0.0

print("Dtypes after casting:")
print(target_metrics.dtypes)
print("\nDescriptive statistics (normalised cols):")
norm_col_names = [c for c in target_metrics.columns if c.endswith("_norm")]
if norm_col_names:
    print(target_metrics[norm_col_names].describe())

In [ ]:
# ── Save cleaned datasets ────────────────────────────────────────────
out_dir = PROJECT_ROOT / "data" / "cleaned"
out_dir.mkdir(parents=True, exist_ok=True)

target_metrics.to_parquet(out_dir / "target_metrics.parquet", index=False)
file_metrics.to_parquet(out_dir / "file_metrics.parquet", index=False)
edge_list.to_parquet(out_dir / "edge_list.parquet", index=False)

print(f"Saved cleaned datasets to {out_dir}")
print(f"  target_metrics : {len(target_metrics)} rows")
print(f"  file_metrics   : {len(file_metrics)} rows")
print(f"  edge_list      : {len(edge_list)} rows")